# Extração de QA dos chats comerciais

Cada conversa de avaliação (ChatGPT web ou Claude web) foi salva como `.html` em `../resultados_chats/`. Este notebook extrai os pares `(pergunta, resposta)` de cada HTML e grava em `.csv` + `.xlsx` ao lado.

**Estrutura do diretório:**

```
resultados_chats/
  {empresa}_std_{provider}.html        ← entrada
  {empresa}_std_{provider}_qa.csv      ← saída (gerada aqui)
  {empresa}_std_{provider}_qa.xlsx     ← saída (gerada aqui)
```

Com `empresa ∈ {bndes, cvm, petro}` e `provider ∈ {chatgpt, claude}` → 6 combinações.

**Diferença Claude vs. ChatGPT** (única coisa que muda entre os dois):
- **Claude web**: mensagens identificadas por `data-testid="user-message"` (perguntas) e `data-is-streaming="false"` contendo `.font-claude-response` (respostas). Limpeza de texto simples.
- **ChatGPT web**: mensagens identificadas por `data-message-author-role ∈ {user, assistant}`. Limpeza precisa filtrar ruídos de UI vazados (`"Você disse:"`, `"O ChatGPT disse:"`, `"Copiar"`, `"Editar"`, etc).

Em ambos os casos, a **primeira pergunta é descartada** (instrução de warmup que não faz parte do benchmark) e os `id`s são renumerados de `1..N`.

**Fluxo do notebook:**

1. Extração de todos os HTMLs para um `dict` em memória (`qas`).
2. Verificação célula a célula contra os `.csv` já existentes — sem gravar nada.
3. Sobrescrita dos `.csv` e `.xlsx` (guardada por flag explícita).

In [1]:
from pathlib import Path
import re
import pandas as pd
from bs4 import BeautifulSoup

RESULTADOS_DIR = Path("../resultados_chats")

EMPRESAS  = ["bndes", "cvm", "petro"]
PROVIDERS = ["chatgpt", "claude"]


# --- Limpeza de texto -----------------------------------------------------

def _clean_text_claude(el):
    """Claude: get_text simples + colapso de quebras + remoção de \n antes de pontuação."""
    txt = el.get_text("\n", strip=True)
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    txt = re.sub(r"\n([,.;:!?])", r"\1", txt)
    return txt.strip()


# Linhas de UI do ChatGPT que vazam pro texto extraído
_CHATGPT_NOISE = {
    "Você disse:",
    "O ChatGPT disse:",
    "Mostrar mais",
    "Mostrar menos",
    "Copiar",
    "Editar",
    "PDF",
}


def _clean_text_chatgpt(el):
    """ChatGPT: mesma limpeza do Claude + filtro linha a linha do ruído de UI."""
    lines = []
    for line in el.get_text("\n", strip=True).splitlines():
        line = line.strip()
        if line and line not in _CHATGPT_NOISE:
            lines.append(line)
    txt = "\n".join(lines)
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    txt = re.sub(r"\n([,.;:!?])", r"\1", txt)
    return txt.strip()


# --- Extração de mensagens do DOM -----------------------------------------

def _extract_messages_claude(soup):
    """Varre o DOM procurando user-message (testid) e respostas do Claude."""
    messages = []
    for el in soup.find_all(True):
        if el.get("data-testid") == "user-message":
            messages.append({"autor": "user", "texto": _clean_text_claude(el)})
        elif el.get("data-is-streaming") == "false":
            resp = el.select_one(".font-claude-response")
            if resp:
                messages.append({"autor": "assistant", "texto": _clean_text_claude(resp)})
    return messages


def _extract_messages_chatgpt(soup):
    """ChatGPT marca cada mensagem com data-message-author-role."""
    messages = []
    for msg in soup.select("[data-message-author-role]"):
        role = msg.get("data-message-author-role")
        if role in {"user", "assistant"}:
            messages.append({"autor": role, "texto": _clean_text_chatgpt(msg)})
    return messages


_EXTRACTORS = {
    "claude":  _extract_messages_claude,
    "chatgpt": _extract_messages_chatgpt,
}


# --- Função pública -------------------------------------------------------

def extrair_qa(html_path: Path, provider: str) -> pd.DataFrame:
    """
    Extrai pares (pergunta, resposta) de uma conversa salva como HTML.

    - A primeira pergunta/resposta é descartada (warmup).
    - Os ids são renumerados de 1..N depois do descarte.
    - Retorna DataFrame com colunas [id, pergunta, resposta].
    """
    html = html_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    messages = _EXTRACTORS[provider](soup)

    rows = []
    i = 0
    while i < len(messages):
        if messages[i]["autor"] != "user":
            i += 1
            continue
        pergunta = messages[i]
        if i + 1 < len(messages) and messages[i + 1]["autor"] == "assistant":
            resposta = messages[i + 1]
            i += 2
        else:
            resposta = None
            i += 1
        rows.append({
            "pergunta": pergunta["texto"],
            "resposta": resposta["texto"] if resposta else "",
        })

    df = pd.DataFrame(rows)
    df = df.iloc[1:].reset_index(drop=True)         # descarta warmup
    df.insert(0, "id", range(1, len(df) + 1))       # renumera ids
    return df


## Etapa 1 — Extração

Roda `extrair_qa` em cada um dos 6 HTMLs e guarda os DataFrames num `dict` indexado por `(empresa, provider)`. Nada é gravado em disco aqui.

In [2]:
qas = {}  # {(empresa, provider): DataFrame}

for empresa in EMPRESAS:
    for provider in PROVIDERS:
        html_path = RESULTADOS_DIR / f"{empresa}_std_{provider}.html"
        if not html_path.exists():
            print(f"[SKIP] {html_path.name} não existe")
            continue
        df = extrair_qa(html_path, provider)
        qas[(empresa, provider)] = df
        print(f"[OK]   {empresa:5}/{provider:7} → {len(df)} pares (id {df['id'].min()}..{df['id'].max()})")

print(f"\nTotal: {len(qas)} conversa(s) processada(s)")


[OK]   bndes/chatgpt → 50 pares (id 1..50)
[OK]   bndes/claude  → 50 pares (id 1..50)
[OK]   cvm  /chatgpt → 50 pares (id 1..50)
[OK]   cvm  /claude  → 50 pares (id 1..50)
[OK]   petro/chatgpt → 50 pares (id 1..50)
[OK]   petro/claude  → 50 pares (id 1..50)

Total: 6 conversa(s) processada(s)


## Etapa 2 — Verificação contra os arquivos existentes

Para cada `(empresa, provider)`, lê o `_qa.csv` que já está no disco e compara com o DataFrame que acabamos de gerar.

**Critério:** mesma quantidade de linhas, mesmas colunas (`id, pergunta, resposta`), mesmo conteúdo célula a célula.

**Cuidado com NaN/string vazia no CSV:** strings vazias gravadas viram `NaN` quando relidas com o pandas padrão. Usamos `keep_default_na=False` na leitura pra não ter falso positivo de diferença.

In [3]:
def comparar_dfs(df_novo: pd.DataFrame, df_velho: pd.DataFrame):
    """Retorna (igual: bool, msg: str, diffs: list[(idx, col, novo, velho)])."""
    if len(df_novo) != len(df_velho):
        return False, f"tamanhos diferentes: novo={len(df_novo)} vs velho={len(df_velho)}", []
    if list(df_novo.columns) != list(df_velho.columns):
        return False, f"colunas: novo={list(df_novo.columns)} vs velho={list(df_velho.columns)}", []

    diffs = []
    for idx in df_novo.index:
        for col in df_novo.columns:
            v_novo  = df_novo.at[idx, col]
            v_velho = df_velho.at[idx, col]
            if str(v_novo) != str(v_velho):
                diffs.append((idx, col, v_novo, v_velho))

    if not diffs:
        return True, "idênticos", []
    return False, f"{len(diffs)} célula(s) diferente(s)", diffs


todos_iguais = True

for (empresa, provider), df_novo in qas.items():
    csv_path = RESULTADOS_DIR / f"{empresa}_std_{provider}_qa.csv"
    tag = f"{empresa:5}/{provider:7}"

    if not csv_path.exists():
        todos_iguais = False
        print(f"[{tag}] CSV existente não encontrado — seria criação nova")
        continue

    df_velho = pd.read_csv(csv_path, encoding="utf-8-sig", keep_default_na=False)
    # `id` lido como int para casar com o df_novo
    df_velho["id"] = df_velho["id"].astype(int)

    igual, msg, diffs = comparar_dfs(df_novo, df_velho)
    if igual:
        print(f"[{tag}] ✓ idêntico ({len(df_novo)} linhas)")
    else:
        todos_iguais = False
        print(f"[{tag}] ✗ {msg}")
        for idx, col, v_n, v_v in diffs[:3]:
            print(f"     linha {idx} coluna {col!r}")
            print(f"       novo  : {str(v_n)[:100]!r}")
            print(f"       velho : {str(v_v)[:100]!r}")
        if len(diffs) > 3:
            print(f"     ... e mais {len(diffs) - 3} diferença(s)")

print()
print("✓ Tudo bate — pode sobrescrever na Etapa 3." if todos_iguais
      else "✗ Há diferenças — investigue antes de sobrescrever.")


[bndes/chatgpt] ✓ idêntico (50 linhas)
[bndes/claude ] ✓ idêntico (50 linhas)
[cvm  /chatgpt] ✓ idêntico (50 linhas)
[cvm  /claude ] ✓ idêntico (50 linhas)
[petro/chatgpt] ✓ idêntico (50 linhas)
[petro/claude ] ✓ idêntico (50 linhas)

✓ Tudo bate — pode sobrescrever na Etapa 3.


## Etapa 3 — Sobrescrita (opcional)

Só rode se a Etapa 2 confirmou que está tudo idêntico (caso em que sobrescrever é no-op seguro) **ou** se você verificou as diferenças e quer mesmo aplicar a nova versão.

Mude `CONFIRMAR_SOBRESCRITA = True` e execute.

In [5]:
CONFIRMAR_SOBRESCRITA = True   # mude para True quando estiver pronto

if not CONFIRMAR_SOBRESCRITA:
    print("CONFIRMAR_SOBRESCRITA=False — nada foi gravado.")
else:
    for (empresa, provider), df in qas.items():
        csv_path  = RESULTADOS_DIR / f"{empresa}_std_{provider}_qa.csv"
        xlsx_path = RESULTADOS_DIR / f"{empresa}_std_{provider}_qa.xlsx"
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        df.to_excel(xlsx_path, index=False)
        print(f"[GRAVADO] {csv_path.name} + {xlsx_path.name}")


[GRAVADO] bndes_std_chatgpt_qa.csv + bndes_std_chatgpt_qa.xlsx
[GRAVADO] bndes_std_claude_qa.csv + bndes_std_claude_qa.xlsx
[GRAVADO] cvm_std_chatgpt_qa.csv + cvm_std_chatgpt_qa.xlsx
[GRAVADO] cvm_std_claude_qa.csv + cvm_std_claude_qa.xlsx
[GRAVADO] petro_std_chatgpt_qa.csv + petro_std_chatgpt_qa.xlsx
[GRAVADO] petro_std_claude_qa.csv + petro_std_claude_qa.xlsx
